# Baseline Model: TF-IDF + SVM Pipeline

This notebook implements the non-Deep Learning solution proposed for the ASHO-AI ICD-10 codification task.

**Objective:** Predict the ICD code **category** (first digit/character) for each clinical literal.

**Pipeline overview:**
1. **Preprocessing (Normalisation):** Lowercase, strip accents, remove punctuation/digits.
2. **Feature Extraction (TF-IDF):** Character n-grams with TF-IDF weighting.
3. **Classification (SVM):** Linear SVM for multi-class single-label classification.
4. **Prediction:** Generate leaderboard submission with columns `id`, `Literal`, `y_category`.

## 1. Setup

In [1]:
import sys, os, time, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make the src module importable from the notebooks folder
sys.path.insert(0, os.path.abspath('../src'))

from data_processing import (
    normalize_text, normalize_texts,
    extract_category,
    prepare_category_dataset, split_category_dataset,
)
from evaluation import (
    calculate_category_metrics, print_classification_report,
    print_comparison_table, plot_comparison,
    generate_submission,
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

RANDOM_STATE = 42
print('Setup complete.')

Setup complete.


## 2. Data Loading

In [2]:
codif_df = pd.read_csv('../data/codification_data.csv')
lead_df  = pd.read_csv('../data/leaderboard_data.csv')

print(f'Training rows  : {len(codif_df):,}')
print(f'Unique codes   : {codif_df["Code"].nunique():,}')
print(f'Unique literals: {codif_df["Literal"].nunique():,}')
print(f'Leaderboard    : {len(lead_df):,}')
print()
display(codif_df.head())
display(lead_df.head())

Training rows  : 13,700
Unique codes   : 4,059
Unique literals: 11,584
Leaderboard    : 6,667



,Code,Literal
0,J9809,Hiperreactividad bronquial
1,J9801,broncoespástica
2,I420,miocardiopatía dilatada
3,Y831,HTA irc 6
4,R5600,Crisis febriles atípicas


,id,Literal
0,1,AMNIODRENAJE
1,2,Hiperparatiroidismo primario
2,3,MIGRANYA parto
3,4,VHC
4,5,Absceso mama izq


## 3. Phase 1 — Preprocessing (Normalisation)

Following the proposal:
- **Lowercasing**
- **Accent stripping** (á→a, ñ→n, etc.)
- **Punctuation & digit handling**
- **Whitespace trimming**

We use `normalize_text()` from `src/data_processing.py`.

In [3]:
# Show normalisation effect on sample texts
samples = [
    'Hiperreactividad bronquial',
    'broncoesp\u00e1stica',
    'miocardiopat\u00eda dilatada',
    'Crisis febriles at\u00edpicas',
    'pr\u00f3tesis mama',
    'ALERGIA IBUPROFENO',
    '<font>VHC</font>',
    'H\u00e8rnia ventral',
]

print(f'{"Original":<35} {"Normalised"}')
print('-' * 65)
for s in samples:
    print(f'{s:<35} {normalize_text(s)}')

Original                            Normalised
-----------------------------------------------------------------
Hiperreactividad bronquial          hiperreactividad bronquial
broncoespástica                     broncoespastica
miocardiopatía dilatada             miocardiopatia dilatada
Crisis febriles atípicas            crisis febriles atipicas
prótesis mama                       protesis mama
ALERGIA IBUPROFENO                  alergia ibuprofeno
<font>VHC</font>                    vhc
Hèrnia ventral                      hernia ventral


## 4. Category Extraction & Dataset Preparation

The target label is the **first character** of the ICD code (the category).

For literals that map to multiple codes with different first characters, we take the **majority-vote** category.

In [4]:
# Show what category extraction looks like
print('Code -> Category examples:')
for code in ['J9809', '07CP0ZZ', 'N801', 'Z6740', 'O99284', 'BW03ZZZ']:
    print(f'  {code:<10} -> {extract_category(code)}')

Code -> Category examples:
  J9809      -> J
  07CP0ZZ    -> 0
  N801       -> N
  Z6740      -> Z
  O99284     -> O
  BW03ZZZ    -> B


In [5]:
# Prepare single-label category dataset
cat_df = prepare_category_dataset(
    codif_df,
    literal_col='Literal',
    code_col='Code',
)
print()
display(cat_df.head(10))

Total rows: 13,700
Unique categories: 36  →  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


Unique literals: 11,584
Category distribution:
y_category
0     967
1     261
2     331
3     498
4     341
5     351
6     492
7     322
8     351
9     239
A      20
B     494
C     188
D     259
E     388
F     245
G     180
H     137
I     313
J     201
K     323
L      94
M     263
N     455
O    1169
P      94
Q     123
R     318
S      69
T      62
U      12
V     414
W       7
X       9
Y      33
Z    1561



,Literal,y_category
0,", Ex fumador",Z
1,", color: teñido",O
2,", se extrae coagulo",1
3,- Grupo sanguíneo: A Positivo,Z
4,- Rx tórax y abdomen,B
5,. A negativo,Z
6,. A positivo,Z
7,. Desgarro I,O
8,. Hernia hiatal,K
9,0 NEGATIVO,Z


In [6]:
# Apply text normalisation
cat_df['Literal'] = normalize_texts(cat_df['Literal'])

# Sanity check
print('After normalisation:')
display(cat_df.head(10))

After normalisation:


,Literal,y_category
0,ex fumador,Z
1,color tenido,O
2,se extrae coagulo,1
3,grupo sanguineo a positivo,Z
4,rx torax y abdomen,B
5,a negativo,Z
6,a positivo,Z
7,desgarro i,O
8,hernia hiatal,K
9,0 negativo,Z


In [7]:
# Stratified train/val split
X_train, X_val, y_train, y_val = split_category_dataset(
    cat_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

Train: 9,267 | Val: 2,317


## 5. Phase 2 — Feature Extraction (TF-IDF)

Character n-grams `(3, 6)` capture typos, abbreviations, and morphological roots in the short Spanish medical texts.

In [8]:
tfidf = TfidfVectorizer(
    analyzer='char_wb',        # character n-grams respecting word boundaries
    ngram_range=(3, 6),        # trigrams to 6-grams
    sublinear_tf=True,         # log-normalised TF
    max_features=100_000,      # keep top features
    min_df=2,                  # ignore very rare n-grams
    dtype=np.float32,          # save memory
)

print('Fitting TF-IDF on training data...')
t0 = time.time()
X_train_tfidf = tfidf.fit_transform(X_train)
t1 = time.time()
print(f'  Done in {t1 - t0:.1f}s')
print(f'  Feature matrix : {X_train_tfidf.shape}')
print(f'  Vocabulary size: {len(tfidf.vocabulary_):,}')

# Transform validation set (do NOT refit)
X_val_tfidf = tfidf.transform(X_val)
print(f'  Val matrix     : {X_val_tfidf.shape}')

Fitting TF-IDF on training data...


  Done in 0.2s
  Feature matrix : (9267, 25667)
  Vocabulary size: 25,667
  Val matrix     : (2317, 25667)


## 6. Phase 3 — Classification (Linear SVM)

Linear SVMs handle high-dimensional sparse TF-IDF features very well.  
`LinearSVC` performs multi-class classification natively (one-vs-rest internally).

In [9]:
print('Training LinearSVC...')
t0 = time.time()

svm_clf = LinearSVC(
    C=1.0,
    max_iter=10_000,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
svm_clf.fit(X_train_tfidf, y_train)

t1 = time.time()
print(f'  Done in {t1 - t0:.1f}s')
print(f'  Classes learned: {len(svm_clf.classes_)}')

Training LinearSVC...


  Done in 0.7s
  Classes learned: 36


### Evaluation on Validation Set

The official metric is **strict accuracy** (exact match of the first digit).

In [10]:
y_pred_val = svm_clf.predict(X_val_tfidf)

metrics = calculate_category_metrics(y_val, y_pred_val)

print('--- SVM Validation Results ---')
for k, v in metrics.items():
    print(f'  {k:<25}: {v:.4f}')

--- SVM Validation Results ---
  accuracy                 : 0.5693
  weighted_f1              : 0.5658
  macro_f1                 : 0.5091
  weighted_precision       : 0.5759
  weighted_recall          : 0.5693


In [11]:
# Per-class breakdown
print('Per-class classification report:')
print_classification_report(y_val, y_pred_val)

Per-class classification report:
              precision    recall  f1-score   support

           0       0.71      0.63      0.67       193
           1       0.57      0.62      0.59        52
           2       0.36      0.24      0.29        66
           3       0.55      0.41      0.47       100
           4       0.47      0.38      0.42        68
           5       0.33      0.30      0.31        70
           6       0.49      0.35      0.41        98
           7       0.21      0.16      0.18        64
           8       0.57      0.46      0.51        70
           9       0.38      0.54      0.44        48
           A       0.25      0.25      0.25         4
           B       0.72      0.78      0.75        99
           C       0.53      0.66      0.59        38
           D       0.51      0.71      0.59        52
           E       0.51      0.54      0.52        78
           F       0.55      0.82      0.66        49
           G       0.57      0.81      0.67     

## 7. Phase 4 — Leaderboard Prediction

Apply the same normalisation and TF-IDF transformation (without refitting) to the leaderboard data, then predict.

In [12]:
# Normalise leaderboard literals
lead_literals_norm = normalize_texts(lead_df['Literal'].tolist())

# Transform with the ALREADY FITTED TF-IDF vectoriser
X_lead_tfidf = tfidf.transform(lead_literals_norm)
print(f'Leaderboard TF-IDF matrix: {X_lead_tfidf.shape}')

# Predict categories
y_lead_pred = svm_clf.predict(X_lead_tfidf)
print(f'Predictions: {len(y_lead_pred):,} literals')
print(f'Unique categories predicted: {len(set(y_lead_pred))}')
print(f'Any empty: {sum(1 for p in y_lead_pred if not p)}')

Leaderboard TF-IDF matrix: (6667, 25667)
Predictions: 6,667 literals
Unique categories predicted: 36
Any empty: 0


In [13]:
# Generate the submission file
os.makedirs('../submissions', exist_ok=True)

submission_df = generate_submission(
    lead_df,
    y_lead_pred,
    output_path='../submissions/svm_baseline.csv',
)

print()
print('Submission preview:')
display(submission_df.head(10))
print()
print('Category distribution in submission:')
print(submission_df['y_category'].value_counts().sort_index())

Submission saved to: ../submissions/svm_baseline.csv
  Total rows: 6,667
  Unique categories predicted: 36
  Any empty values: 0

Submission preview:


,id,Literal,y_category
0,1,AMNIODRENAJE,1
1,2,Hiperparatiroidismo primario,E
2,3,MIGRANYA parto,G
3,4,VHC,B
4,5,Absceso mama izq,N
5,6,miomas parto,O
6,7,cos estrany,9
7,8,TCE,S
8,9,osteogénesis imperfecta,Q
9,10,Soplo sistólico,R



Category distribution in submission:
y_category
0    539
1    133
2    153
3    205
4    159
5    164
6    199
7    110
8    148
9    205
A     22
B    291
C    128
D    227
E    302
F    235
G    140
H     90
I    223
J    185
K    244
L     73
M    168
N    310
O    629
P     66
Q     77
R    195
S     40
T     35
U      2
V    231
W      9
X      7
Y     33
Z    690
Name: count, dtype: int64


## 8. Summary

| Step | Detail |
|------|--------|
| **Preprocessing** | Lowercased, stripped accents, removed punctuation/digits, collapsed whitespace |
| **Features** | Character n-grams (3–6) with TF-IDF, 100k max features |
| **Model** | LinearSVC, class_weight='balanced', multi-class (one-vs-rest) |
| **Target** | ICD code category (first character) — single-label |
| **Evaluation** | Strict accuracy (exact match on first digit) |
| **Output** | `submissions/svm_baseline.csv` — columns: `id`, `Literal`, `y_category` |